# Agente Inteligente para Consulta de Arquivos CSV

## Objetivo
Desenvolver um agente inteligente capaz de responder perguntas em linguagem natural sobre dados armazenados em arquivos CSV.

### Tecnologias Utilizadas
- **LLM**: Groq (API)
- **Framework de Agentes**: LangChain
- **Processamento de Dados**: Pandas
- **Visualização**: Plotly
- **Interface**: IPython widgets (para notebook) + funções auxiliares

### Arquitetura da Solução
```
┌─────────────────┐
│  Upload ZIP     │  Interface A: Carga de dados
└────────┬────────┘
         │
         ▼
┌─────────────────────────────┐
│  Processamento de CSV       │  Extração e catalogação
│  + Dicionário de dados      │
└────────┬────────────────────┘
         │
         ▼
┌─────────────────────────────┐
│  Armazenamento em memória   │  DataFrames + metadados
│  (ou banco de dados)        │
└────────┬────────────────────┘
         │
         ▼
┌─────────────────────────────────────────┐
│         Agente Inteligente (Groq)       │  Interface B: Consultas
│  - Parser de perguntas                  │
│  - Executor de análises (Tools)         │
│  - Gerador de visualizações             │
└─────────────────────────────────────────┘
         │
         ▼
┌──────────────────────────────┐
│  Respostas (Texto/Tabela/   │
│  Gráfico)                    │
└──────────────────────────────┘
```


## 1. Instalação de Dependências

In [ ]:
# Instale as dependências necessárias
# Descomente a próxima linha apenas na primeira execução

#!pip install langchain langchain-groq pandas plotly python-dotenv ipywidgets

In [2]:
# %pip install --upgrade plotly

In [ ]:
# Comentando para não executar todas as vezes
# %pip install --upgrade ipywidgets

In [ ]:
# Comentando para não executar todas as vezes
# %pip install --upgrade create_agent

In [ ]:
# Comentando para não executar todas as vezes
# %pip install langchain-classic

## 2. Imports e Configurações Iniciais

In [1]:
# ============================================================================
# IMPORTS E CONFIGURAÇÃO
# ============================================================================

import io
import json
import os
import tempfile
import warnings
import zipfile
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from ipywidgets import Button, FileUpload, HBox, HTML as HTMLWidget, Label, Output, Text, Textarea, VBox
from langchain.agents import create_agent as create_langchain_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import StructuredTool, tool
from langchain_groq import ChatGroq
import plotly.express as px
import plotly.graph_objects as go
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

print("✓ Todos os imports carregados com sucesso!")

✓ Todos os imports carregados com sucesso!


## 3. Configuração da API Groq

In [40]:
# Configure sua chave de API do Groq
# Opção 1: Defina a variável de ambiente
# os.environ["GROQ_API_KEY"] = "sua-chave-aqui"

# Opção 2: Carregue de um arquivo .env
from dotenv import load_dotenv
load_dotenv()

# Opção 3: Use a célula abaixo para inserir a chave interativamente
if not os.environ.get("GROQ_API_KEY"):
    print("⚠️  GROQ_API_KEY não configurada.")
    print("\nAções:")
    print("1. Defina a variável no código acima (Opção 1)")
    print("2. Crie um arquivo .env com GROQ_API_KEY=sua-chave-aqui")
    print("3. Use a célula abaixo para configurar interativamente:")
    
# Inicialize o cliente Groq
try:
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",  # Modelo rápido e eficiente
        temperature=0,  # Determinístico para análise de dados
        max_tokens=4000,
        api_key=os.environ.get("GROQ_API_KEY")
    )
    print("✓ Cliente Groq inicializado com sucesso!")
except Exception as e:
    print(f"✗ Erro ao inicializar Groq: {e}")
    llm = None

✓ Cliente Groq inicializado com sucesso!


## 4. Armazenamento Global de Dados

Este módulo gerencia os DataFrames carregados e seus metadados.

In [41]:
class DataManager:
    """Gerenciador centralizado de dados CSV carregados."""
    
    def __init__(self):
        self.dataframes: Dict[str, pd.DataFrame] = {}
        self.data_dictionary: Dict[str, Dict[str, str]] = {}
        self.metadata: Dict[str, Dict[str, Any]] = {}
    
    def add_dataframe(self, name: str, df: pd.DataFrame, description: str = ""):
        """Adiciona um DataFrame ao gerenciador."""
        self.dataframes[name] = df
        self.metadata[name] = {
            "rows": len(df),
            "columns": len(df.columns),
            "description": description,
            "dtypes": df.dtypes.to_dict()
        }
    
    def add_data_dictionary(self, dataframe_name: str, dictionary: Dict[str, str]):
        """Adiciona um dicionário de dados (descrição das colunas)."""
        self.data_dictionary[dataframe_name] = dictionary
    
    def get_dataframe(self, name: str) -> Optional[pd.DataFrame]:
        """Retorna um DataFrame específico."""
        return self.dataframes.get(name)
    
    def get_all_names(self) -> List[str]:
        """Retorna lista de nomes de DataFrames carregados."""
        return list(self.dataframes.keys())
    
    def get_summary(self) -> str:
        """Retorna um resumo de todos os dados carregados."""
        if not self.dataframes:
            return "Nenhum dado carregado."
        
        summary = "📊 **Dados Carregados:**\n\n"
        for name, df in self.dataframes.items():
            meta = self.metadata[name]
            summary += f"- **{name}**: {meta['rows']} linhas × {meta['columns']} colunas\n"
            summary += f"  Colunas: {', '.join(df.columns.tolist())}\n\n"
        return summary
    
    def clear(self):
        """Limpa todos os dados carregados."""
        self.dataframes.clear()
        self.data_dictionary.clear()
        self.metadata.clear()

# Instância global
data_manager = DataManager()
print("✓ DataManager inicializado!")

✓ DataManager inicializado!


## 5. Processador de Arquivos ZIP

In [42]:
class ZipProcessor:
    """Processa arquivos ZIP contendo CSVs e dicionário de dados."""
    
    @staticmethod
    def process_zip(zip_path: str) -> tuple[bool, str, Dict[str, pd.DataFrame]]:
        """
        Processa um arquivo ZIP e extrai CSVs e dicionário de dados.
        
        Args:
            zip_path: Caminho para o arquivo ZIP
            
        Returns:
            (sucesso, mensagem, dataframes)
        """
        dataframes = {}
        data_dict = {}
        
        try:
            with tempfile.TemporaryDirectory() as temp_dir:
                # Extrai o ZIP
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(temp_dir)
                
                # Procura por arquivos CSV e JSON (dicionário)
                temp_path = Path(temp_dir)
                
                # Carrega CSVs
                csv_files = list(temp_path.glob('**/*.csv'))
                if not csv_files:
                    return False, "❌ Nenhum arquivo CSV encontrado no ZIP", {}
                
                for csv_file in csv_files:
                    try:
                        df = pd.read_csv(csv_file, encoding='utf-8')
                        name = csv_file.stem  # Nome sem extensão
                        dataframes[name] = df
                    except Exception as e:
                        return False, f"❌ Erro ao ler {csv_file.name}: {str(e)}", {}
                
                # Carrega dicionário de dados (se existir)
                json_files = list(temp_path.glob('**/*.json'))
                for json_file in json_files:
                    try:
                        with open(json_file, 'r', encoding='utf-8') as f:
                            data_dict = json.load(f)
                    except Exception as e:
                        print(f"⚠️  Aviso ao ler {json_file.name}: {str(e)}")
                
                msg = f"✓ {len(csv_files)} arquivo(s) CSV carregado(s) com sucesso!\n"
                for name in dataframes:
                    msg += f"  - {name}: {len(dataframes[name])} linhas\n"
                
                return True, msg, dataframes, data_dict
        
        except Exception as e:
            return False, f"❌ Erro ao processar ZIP: {str(e)}", {}, {}

print("✓ ZipProcessor criado!")

✓ ZipProcessor criado!


In [ ]:
# Comentando para não executar todas as vezes
# %pip install -U langchain-core langgraph langchain   

^C
Note: you may need to restart the kernel to use updated packages.


## 6. Ferramentas (Tools) para o Agente

Essas ferramentas permitem que o agente realize análises de dados.

In [43]:
@tool
def list_available_data() -> str:
    """
    Lista todos os arquivos CSV e suas colunas disponíveis para análise.
    Use esta ferramenta para entender quais dados estão disponíveis.
    """
    summary = "Dados disponíveis:\n\n"
    for name, df in data_manager.dataframes.items():
        meta = data_manager.metadata[name]
        summary += f"📄 {name}\n"
        summary += f"  Dimensões: {meta['rows']} linhas × {meta['columns']} colunas\n"
        summary += f"  Colunas: {', '.join(df.columns.tolist())}\n"
        summary += f"  Tipos: {dict(df.dtypes)}\n\n"
    return summary if summary != "Dados disponíveis:\n\n" else "Nenhum dado carregado."

@tool
def get_dataframe_info(dataframe_name: str) -> str:
    """
    Obtém informações detalhadas sobre um DataFrame específico.
    Retorna estatísticas descritivas e informações sobre valores ausentes.
    """
    df = data_manager.get_dataframe(dataframe_name)
    if df is None:
        return f"Erro: DataFrame '{dataframe_name}' não encontrado."
    
    info = f"Informações de '{dataframe_name}':\n\n"
    info += f"Shape: {df.shape}\n\n"
    info += f"Tipos de dados:\n{df.dtypes.to_string()}\n\n"
    info += f"Valores ausentes:\n{df.isnull().sum()}\n\n"
    info += f"Estatísticas descritivas:\n{df.describe().to_string()}"
    return info

@tool
def query_dataframe(dataframe_name: str, operation: str) -> str:
    """
    Executa operações de análise em um DataFrame.
    Operações suportadas:
    - sum: Sum de colunas numéricas
    - mean: Média de colunas numéricas
    - groupby_sum: Agrupamento com soma (formato: "groupby_sum:coluna_grupo:coluna_valor")
    - groupby_mean: Agrupamento com média
    - groupby_count: Agrupamento com contagem
    - top_n: Top N valores (formato: "top_n:coluna:n")
    - filter: Filtrar dados (formato: "filter:coluna:valor")
    """
    df = data_manager.get_dataframe(dataframe_name)
    if df is None:
        return f"Erro: DataFrame '{dataframe_name}' não encontrado."
    
    try:
        parts = operation.split(":")
        op_type = parts[0].lower()
        
        if op_type == "sum":
            result = df.select_dtypes(include=[np.number]).sum()
            return f"Soma total:\n{result.to_string()}"
        
        elif op_type == "mean":
            result = df.select_dtypes(include=[np.number]).mean()
            return f"Média:\n{result.to_string()}"
        
        elif op_type == "groupby_sum" and len(parts) >= 3:
            group_col = parts[1]
            value_col = parts[2]
            result = df.groupby(group_col)[value_col].sum().sort_values(ascending=False)
            return f"Agrupamento por {group_col} (soma de {value_col}):\n{result.to_string()}"
        
        elif op_type == "groupby_mean" and len(parts) >= 3:
            group_col = parts[1]
            value_col = parts[2]
            result = df.groupby(group_col)[value_col].mean().sort_values(ascending=False)
            return f"Agrupamento por {group_col} (média de {value_col}):\n{result.to_string()}"
        
        elif op_type == "groupby_count" and len(parts) >= 2:
            group_col = parts[1]
            result = df[group_col].value_counts()
            return f"Contagem por {group_col}:\n{result.to_string()}"
        
        elif op_type == "top_n" and len(parts) >= 3:
            col = parts[1]
            n = int(parts[2])
            result = df.nlargest(n, col)[[col]]
            return f"Top {n} valores de {col}:\n{result.to_string()}"
        
        else:
            return "Operação não suportada. Use: sum, mean, groupby_sum, groupby_mean, groupby_count, ou top_n"
    
    except Exception as e:
        return f"Erro ao executar operação: {str(e)}"

@tool
def get_unique_values(dataframe_name: str, column_name: str) -> str:
    """
    Retorna os valores únicos de uma coluna (útil para entender categorias).
    """
    df = data_manager.get_dataframe(dataframe_name)
    if df is None:
        return f"Erro: DataFrame '{dataframe_name}' não encontrado."
    
    if column_name not in df.columns:
        return f"Erro: Coluna '{column_name}' não encontrada."
    
    unique_vals = df[column_name].unique()
    return f"Valores únicos em {column_name} ({len(unique_vals)} total):\n{list(unique_vals)}"

@tool
def get_sample_data(dataframe_name: str, n_rows: int = 5) -> str:
    """
    Retorna as primeiras N linhas de um DataFrame (para exploração).
    """
    df = data_manager.get_dataframe(dataframe_name)
    if df is None:
        return f"Erro: DataFrame '{dataframe_name}' não encontrado."
    
    return f"Primeiras {n_rows} linhas de {dataframe_name}:\n{df.head(n_rows).to_string()}"

print("✓ Ferramentas (Tools) criadas com sucesso!")

# Lista de tools disponíveis
tools = [
    list_available_data,
    get_dataframe_info,
    query_dataframe,
    get_unique_values,
    get_sample_data
]

✓ Ferramentas (Tools) criadas com sucesso!


In [ ]:
# Comentando para não executar todas as vezes
# %pip install -U langchain langchain-core langchain-community

## 7. Criação do Agente Inteligente

In [44]:
def create_agent():
    """Cria e configura o agente inteligente com Groq."""

    system_prompt = """
Você é um agente de análise de dados especializado em responder perguntas sobre arquivos CSV.

SUAS RESPONSABILIDADES:
1. Entender a pergunta do usuário em linguagem natural
2. Identificar qual(is) DataFrame(s) contém os dados necessários
3. Usar as ferramentas disponíveis para extrair e analisar dados
4. Formular uma resposta clara e útil

INSTRUÇÕES IMPORTANTES:
- SEMPRE comece listando os dados disponíveis se não souber quais arquivos existem
- Para perguntas sobre FORNECEDORES, PRODUTOS, etc., procure por colunas com esses nomes
- Use groupby_sum para somas por grupo (ex: total por fornecedor)
- Use groupby_mean para médias por grupo
- Use top_n para encontrar os maiores/menores valores
- Quando precisar de múltiplas operações, faça-as em sequência
- SEMPRE interprete os resultados e forneça insights, não apenas dados brutos
- Formate as respostas de forma clara e legível

FORMATO DE RESPOSTA:
- Comece com um resumo direto da pergunta
- Apresente os dados analisados
- Forneça insights e conclusões
- Se apropriado, sugira visualizações (gráficos de barras, linhas, pizza, etc.)

Você tem acesso às seguintes ferramentas para análise:
- list_available_data: Lista todos os CSVs carregados
- get_dataframe_info: Informações detalhadas sobre um CSV
- query_dataframe: Realiza operações de análise (sum, mean, groupby, top_n)
- get_unique_values: Mostra valores únicos em uma coluna
- get_sample_data: Mostra primeiras linhas de um CSV
    """

    agent = create_langchain_agent(
        model=llm,
        tools=tools,
        system_prompt=system_prompt,
        debug=False,
    )
    return agent

# Cria o agente
if llm:
    agent_executor = create_agent()
    print("✓ Agente inteligente criado com sucesso!")
    print(type(agent_executor).__name__)
else:
    print("✗ Não foi possível criar o agente (LLM não inicializado)")
    agent_executor = None

✓ Agente inteligente criado com sucesso!
CompiledStateGraph


## 8. Funções de Apoio para Visualização

In [46]:
def create_bar_chart(data: dict, title: str, x_label: str = "Categorias", y_label: str = "Valores"):
    """
    Cria um gráfico de barras interativo.
    
    Args:
        data: Dicionário {label: valor}
        title: Título do gráfico
    """
    fig = go.Figure(data=[go.Bar(
        x=list(data.keys()),
        y=list(data.values()),
        marker_color='lightblue'
    )])
    fig.update_layout(
        title=title,
        xaxis_title=x_label,
        yaxis_title=y_label,
        hovermode='x unified',
        height=400
    )
    return fig

def create_line_chart(data: dict, title: str, x_label: str = "Tempo", y_label: str = "Valores"):
    """
    Cria um gráfico de linhas interativo.
    """
    fig = go.Figure(data=[go.Scatter(
        x=list(data.keys()),
        y=list(data.values()),
        mode='lines+markers',
        line=dict(color='steelblue', width=2),
        marker=dict(size=8)
    )])
    fig.update_layout(
        title=title,
        xaxis_title=x_label,
        yaxis_title=y_label,
        hovermode='x unified',
        height=400
    )
    return fig

def create_pie_chart(data: dict, title: str):
    """
    Cria um gráfico de pizza interativo.
    """
    fig = go.Figure(data=[go.Pie(
        labels=list(data.keys()),
        values=list(data.values())
    )])
    fig.update_layout(
        title=title,
        height=500
    )
    return fig

def create_table_html(df: pd.DataFrame, title: str = "") -> str:
    """
    Cria uma tabela HTML formatada.
    """
    html = f"<h4>{title}</h4>" if title else ""
    html += df.head(20).to_html(classes='table table-striped', index=False)
    return html

print("✓ Funções de visualização criadas!")

✓ Funções de visualização criadas!


## 9. Interface A - Upload e Processamento de Dados

In [ ]:
def create_upload_interface():
    """
    Cria a Interface A: Upload de arquivo ZIP contendo CSVs.
    """
    
    print("="*60)
    print("INTERFACE A - CARREGAMENTO DE DADOS")
    print("="*60)
    print()
    
    # Widget de upload
    file_upload = FileUpload(
        accept='.zip',
        multiple=False,
        description='Selecione arquivo ZIP'
    )
    
    output = Output()
    button = Button(description='📥 Processar', button_style='info')
    clear_button = Button(description='🗑️ Limpar Dados', button_style='warning')
    
    def on_process_click(b):
        """Processa o arquivo ZIP quando o botão é clicado."""
        with output:
            output.clear_output()
            
            if not file_upload.value:
                print("❌ Nenhum arquivo selecionado!")
                return
            
            print("⏳ Processando arquivo...\n")
            
            # Salva o arquivo temporário
            if isinstance(file_upload.value, dict):
                uploaded_filename = list(file_upload.value.keys())[0]
                uploaded_content = file_upload.value[uploaded_filename]['content']
            else:
                uploaded_file = file_upload.value[0]
                uploaded_filename = uploaded_file['name']
                uploaded_content = uploaded_file['content']
            
            with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp_file:
                tmp_file.write(uploaded_content)
                tmp_path = tmp_file.name
            
            # Processa o ZIP
            success, message, dataframes, data_dict = ZipProcessor.process_zip(tmp_path)
            print(message)
            
            if success:
                # Carrega os dados no gerenciador
                for name, df in dataframes.items():
                    data_manager.add_dataframe(name, df)
                    if name in data_dict:
                        data_manager.add_data_dictionary(name, data_dict[name])
                
                print("\n" + data_manager.get_summary())
                print("✅ Dados carregados com sucesso! Você pode agora fazer perguntas na Interface B.")
            
            # Limpa arquivo temporário
            os.remove(tmp_path)
    
    def on_clear_click(b):
        """Limpa os dados carregados."""
        with output:
            output.clear_output()
            data_manager.clear()
            print("🗑️ Todos os dados foram removidos.")
    
    button.on_click(on_process_click)
    clear_button.on_click(on_clear_click)
    
    vbox = VBox([
        Label("Selecione um arquivo ZIP contendo arquivos CSV:"),
        file_upload,
        HBox([button, clear_button]),
        output
    ])
    
    display(vbox)

# Cria a interface
create_upload_interface()

INTERFACE A - CARREGAMENTO DE DADOS



## 10. Interface B - Consulta com Agente Inteligente

In [48]:
class ChatInterface:
    """Interface de chat para consultas ao agente inteligente."""
    
    def __init__(self):
        self.chat_history = []
    
    def query_agent(self, question: str) -> str:
        """
        Envia uma pergunta ao agente e obtém a resposta.
        
        Args:
            question: Pergunta em linguagem natural
            
        Returns:
            Resposta do agente
        """
        if not agent_executor:
            return "❌ Agente não foi inicializado. Verifique a chave GROQ_API_KEY."
        
        if not data_manager.get_all_names():
            return "❌ Nenhum dado carregado. Por favor, carregue um arquivo ZIP primeiro (Interface A)."
        
        try:
            print("inicio do try")
            print("input carregado, chamando agente...")
            
            # Prepara o input no formato esperado pela nova API do LangChain
            # A entrada deve ser um dict com "messages" key contendo lista de mensagens
            # input_data = {
            #     "messages": self.chat_history + [HumanMessage(content=question)]
            # }
            input_data = {"input": question}
            
            print("agent_executor type:", type(agent_executor))
            print("Invocando agente com novo formato de entrada...")
            
            # Invoca o agente com o novo formato
            result = agent_executor.invoke(input_data)
            
            print("Agente invocado com sucesso!")
            print("result type:", type(result))
            
            if result is None:
                return "❌ Erro do agente: resposta vazia (result=None). Verifique a configuração do LLM e da API."
            
            # Extrai a resposta do resultado
            # result deve ser um dict com chave "messages" contendo a lista de mensagens
            answer = None
            
            if isinstance(result, dict):
                messages = result.get("messages", [])
                if messages:
                    # A última mensagem deve ser a resposta do agente
                    last_message = messages[-1]
                    if hasattr(last_message, "content"):
                        answer = last_message.content
                    elif isinstance(last_message, dict):
                        answer = last_message.get("content", str(last_message))
                    else:
                        answer = str(last_message)
                
                if not answer:
                    # Se não encontrou em messages, tenta extrair diretamente do dict
                    answer = (
                        result.get("output")
                        or result.get("output_text")
                        or result.get("text")
                        or result.get("answer")
                        or str(result)
                    )
            else:
                # Se não é dict, tenta extrair de atributos
                answer = (
                    getattr(result, "output", None)
                    or getattr(result, "content", None)
                    or str(result)
                )
            
            print("Resposta extraída com sucesso!")
            
            if answer is None or (isinstance(answer, str) and not answer.strip()):
                return "❌ Erro do agente: resposta vazia (answer=None ou vazia)."
            
            # Atualiza histórico
            self.chat_history.append(HumanMessage(content=question))
            self.chat_history.append(AIMessage(content=str(answer)))
            
            return str(answer)
        
        except Exception as e:
            print(f"❌ Exceção capturada: {str(e)}")
            import traceback
            traceback.print_exc()
            return f"❌ Erro ao processar pergunta: {str(e)}"
    
    def clear_history(self):
        """Limpa o histórico de chat."""
        self.chat_history = []

# Instância global do chat
chat = ChatInterface()

print("✓ ChatInterface criada!")


✓ ChatInterface criada!


## 11. Criar Interface de Chat Interativa

In [ ]:
def create_query_interface():
    """
    Cria a Interface B: Consulta com o agente inteligente.
    """
    
    print("\n" + "="*60)
    print("INTERFACE B - CONSULTA COM AGENTE INTELIGENTE")
    print("="*60)
    print()
    
    # Widgets
    query_input = Textarea(
        placeholder='Digite sua pergunta em linguagem natural...',
        description='Pergunta:',
        rows=3,
        layout={'width': '100%'}
    )
    
    send_button = Button(description='🚀 Enviar Pergunta', button_style='success')
    clear_history_button = Button(description='🔄 Limpar Histórico', button_style='info')
    
    output = Output()
    
    def on_send_click(b):
        """Processa a pergunta quando o botão é clicado."""
        with output:
            output.clear_output()
            
            question = query_input.value.strip()
            if not question:
                print("❌ Por favor, digite uma pergunta!")
                return
            
            print(f"🤔 Processando pergunta: '{question}'\n")
            print("-" * 60)
            
            # Chama o agente
            response = chat.query_agent(question)
            
            print("\n📊 RESPOSTA DO AGENTE:")
            print("-" * 60)
            print(response)
            print("-" * 60)
            
            # Limpa o input
            query_input.value = ""
    
    def on_clear_history(b):
        """Limpa o histórico de chat."""
        with output:
            output.clear_output()
            chat.clear_history()
            print("🔄 Histórico de chat limpo!")
    
    send_button.on_click(on_send_click)
    clear_history_button.on_click(on_clear_history)
    
    # Layout
    vbox = VBox([
        HTMLWidget("<h3>💬 Faça perguntas sobre seus dados</h3>"),
        HTMLWidget("<p>Exemplos de perguntas:</p><ul>"
                   "<li>Qual fornecedor recebeu o maior valor?</li>"
                   "<li>Qual é o total de vendas por mês?</li>"
                   "<li>Quais são os 5 maiores clientes?</li>"
                   "</ul>"),
        query_input,
        HBox([send_button, clear_history_button]),
        output
    ])
    
    display(vbox)

# Cria a interface de consulta
create_query_interface()


INTERFACE B - CONSULTA COM AGENTE INTELIGENTE



## 12. Exemplo de Uso - Testes Programáticos

Esta seção demonstra como usar o sistema sem interface gráfica (útil para testes).

In [50]:
# Exemplo: Criar dados de teste

print("\n" + "="*60)
print("CRIANDO DADOS DE EXEMPLO PARA TESTE")
print("="*60 + "\n")

# Cria um DataFrame de exemplo (Notas Fiscais)
nf_data = {
    'Data': pd.date_range('2024-01-01', periods=20, freq='D'),
    'Fornecedor': ['Fornecedor A', 'Fornecedor B', 'Fornecedor C', 'Fornecedor A', 'Fornecedor B'] * 4,
    'Produto': ['Produto X', 'Produto Y', 'Produto Z', 'Produto X', 'Produto Y'] * 4,
    'Categoria': ['Eletrônicos', 'Software', 'Serviços', 'Eletrônicos', 'Software'] * 4,
    'Quantidade': [10, 5, 3, 15, 8, 12, 4, 6, 9, 11, 7, 13, 5, 8, 14, 6, 10, 9, 12, 4],
    'Valor_Unitario': [100.00, 50.00, 200.00, 100.00, 50.00, 100.00, 50.00, 200.00, 100.00, 50.00,
                       200.00, 100.00, 50.00, 200.00, 100.00, 50.00, 200.00, 100.00, 50.00, 200.00],
    'Valor_Total': [1000, 250, 600, 1500, 400, 1200, 200, 1200, 900, 550,
                    2600, 1300, 250, 1600, 1400, 300, 2000, 900, 600, 800]
}

df_nf = pd.DataFrame(nf_data)

# Adiciona ao DataManager
data_manager.add_dataframe(
    name='Notas_Fiscais',
    df=df_nf,
    description='Dados de notas fiscais de compras'
)

print("✓ Dados de exemplo carregados!")
print("\nPrimeiras linhas:")
print(df_nf.head())


CRIANDO DADOS DE EXEMPLO PARA TESTE

✓ Dados de exemplo carregados!

Primeiras linhas:
        Data    Fornecedor    Produto    Categoria  Quantidade  \
0 2024-01-01  Fornecedor A  Produto X  Eletrônicos          10   
1 2024-01-02  Fornecedor B  Produto Y     Software           5   
2 2024-01-03  Fornecedor C  Produto Z     Serviços           3   
3 2024-01-04  Fornecedor A  Produto X  Eletrônicos          15   
4 2024-01-05  Fornecedor B  Produto Y     Software           8   

   Valor_Unitario  Valor_Total  
0           100.0         1000  
1            50.0          250  
2           200.0          600  
3           100.0         1500  
4            50.0          400  


## 13. Testes com Perguntas Reais

In [51]:
# Teste 1: Qual fornecedor recebeu o maior valor?
print("\n" + "="*70)
print("TESTE 1: Qual fornecedor recebeu o maior valor?")
print("="*70)

if agent_executor and data_manager.get_all_names():
    response = chat.query_agent("Qual fornecedor recebeu o maior valor total?")
    print(response)
else:
    print("❌ Agente não está disponível. Configure a chave GROQ_API_KEY.")


TESTE 1: Qual fornecedor recebeu o maior valor?
inicio do try
input carregado, chamando agente...
agent_executor type: <class 'langgraph.graph.state.CompiledStateGraph'>
Invocando agente com novo formato de entrada...
Agente invocado com sucesso!
result type: <class 'dict'>
Resposta extraída com sucesso!
Resumo: 
A pergunta do usuário foi sobre os dados disponíveis e as informações detalhadas sobre o DataFrame 'Notas_Fiscais'. Os dados disponíveis incluem os DataFrames 'train' e 'Notas_Fiscais'. O DataFrame 'Notas_Fiscais' contém informações sobre notas fiscais, incluindo data, fornecedor, produto, categoria, quantidade, valor unitário e valor total. As estatísticas descritivas mostram que o valor total médio por nota fiscal é de 977,50, com um valor mínimo de 200,00 e um valor máximo de 2600,00. O agrupamento por fornecedor mostra que o Fornecedor A tem o maior valor total, seguido pelo Fornecedor B e pelo Fornecedor C.

Dados analisados:
- O DataFrame 'Notas_Fiscais' contém 20 linha

In [52]:
# Teste 2: Qual produto apresentou maior volume?
print("\n" + "="*70)
print("TESTE 2: Qual produto apresentou maior volume comprado?")
print("="*70)

if agent_executor and data_manager.get_all_names():
    response = chat.query_agent("Qual produto teve o maior volume (quantidade) total comprado?")
    print(response)
else:
    print("❌ Agente não está disponível.")


TESTE 2: Qual produto apresentou maior volume comprado?
inicio do try
input carregado, chamando agente...
agent_executor type: <class 'langgraph.graph.state.CompiledStateGraph'>
Invocando agente com novo formato de entrada...
Agente invocado com sucesso!
result type: <class 'dict'>
Resposta extraída com sucesso!
Resumo: 
A pergunta do usuário foi respondida listando os dados disponíveis, obtendo informações detalhadas sobre o DataFrame 'Notas_Fiscais', identificando os valores únicos na coluna 'Fornecedor' e realizando um agrupamento por 'Fornecedor' com a soma de 'Valor_Total'. 

Os dados disponíveis incluem os DataFrames 'train' e 'Notas_Fiscais'. O DataFrame 'Notas_Fiscais' contém informações sobre notas fiscais, incluindo data, fornecedor, produto, categoria, quantidade, valor unitário e valor total.

As informações detalhadas sobre o DataFrame 'Notas_Fiscais' mostram que ele tem 20 linhas e 7 colunas, com tipos de dados variados, incluindo datetime, string, int64 e float64. Não h

In [53]:
# Teste 3: Total gasto em cada categoria
print("\n" + "="*70)
print("TESTE 3: Qual foi o total gasto em cada categoria?")
print("="*70)

if agent_executor and data_manager.get_all_names():
    response = chat.query_agent("Qual é o total gasto em cada categoria de produtos?")
    print(response)
else:
    print("❌ Agente não está disponível.")


TESTE 3: Qual foi o total gasto em cada categoria?
inicio do try
input carregado, chamando agente...
agent_executor type: <class 'langgraph.graph.state.CompiledStateGraph'>
Invocando agente com novo formato de entrada...
Agente invocado com sucesso!
result type: <class 'dict'>
Resposta extraída com sucesso!
Resumo: 
A pergunta do usuário foi sobre os dados disponíveis nos arquivos CSV. Após listar os dados disponíveis, foi realizada uma análise detalhada do DataFrame 'Notas_Fiscais'. Foram obtidas informações sobre os tipos de dados, valores ausentes e estatísticas descritivas. Além disso, foram identificados os valores únicos na coluna 'Fornecedor' e realizada uma operação de agrupamento por 'Fornecedor' com soma de 'Valor_Total'. 

Os dados analisados mostram que o DataFrame 'Notas_Fiscais' contém 20 linhas e 7 colunas, com dados sobre notas fiscais, incluindo data, fornecedor, produto, categoria, quantidade, valor unitário e valor total. A análise descritiva mostra que a média de q

In [54]:
# Teste 4: Quais são os 3 maiores fornecedores?
print("\n" + "="*70)
print("TESTE 4: Quais são os 3 maiores fornecedores por valor total?")
print("="*70)

if agent_executor and data_manager.get_all_names():
    response = chat.query_agent("Liste os 3 maiores fornecedores pelo valor total de compras.")
    print(response)
else:
    print("❌ Agente não está disponível.")


TESTE 4: Quais são os 3 maiores fornecedores por valor total?
inicio do try
input carregado, chamando agente...
agent_executor type: <class 'langgraph.graph.state.CompiledStateGraph'>
Invocando agente com novo formato de entrada...
Agente invocado com sucesso!
result type: <class 'dict'>
Resposta extraída com sucesso!
A pergunta do usuário foi sobre os dados disponíveis e como eles podem ser analisados. 

Os dados disponíveis incluem dois DataFrames: 'train' e 'Notas_Fiscais'. O DataFrame 'Notas_Fiscais' contém informações sobre notas fiscais, incluindo data, fornecedor, produto, categoria, quantidade, valor unitário e valor total.

As estatísticas descritivas do DataFrame 'Notas_Fiscais' mostram que há 20 linhas e 7 colunas, com tipos de dados variados, incluindo datetime, string, int e float. Não há valores ausentes nos dados.

A análise do DataFrame 'Notas_Fiscais' usando a operação 'groupby_sum' por fornecedor mostra que o Fornecedor A tem o maior valor total, seguido pelo Fornece

## 14. Documentação de Arquitetura

### Componentes Principais

#### 1. **DataManager** (Gerenciador de Dados)
- Armazena DataFrames carregados em memória
- Mantém metadados (número de linhas, colunas, tipos)
- Fornece interface centralizada para acesso aos dados

#### 2. **ZipProcessor** (Processador de Arquivos)
- Extrai arquivos ZIP
- Carrega CSVs e dicionários de dados
- Valida integridade dos dados

#### 3. **Tools (Ferramentas)** - Interface do Agente com Dados
- `list_available_data`: Lista todos os dados disponíveis
- `get_dataframe_info`: Informações detalhadas sobre um DataFrame
- `query_dataframe`: Realiza operações de análise (sum, mean, groupby, top_n)
- `get_unique_values`: Retorna valores únicos de uma coluna
- `get_sample_data`: Mostra primeiras linhas de um DataFrame

#### 4. **Agent (Agente Inteligente)**
- Baseado em LangChain com Groq como LLM
- Usa system prompt para definir comportamento
- Seleciona e executa tools automaticamente
- Interpreta resultados e gera respostas em linguagem natural

#### 5. **ChatInterface** (Interface de Consulta)
- Gerencia conversa com histórico
- Coordena entre usuário e agente
- Apresenta respostas formatadas

### Fluxo de Execução

```
1. Usuário faz pergunta
   ↓
2. ChatInterface envia para Agent
   ↓
3. Agente (Groq) interpreta pergunta
   ↓
4. Agente seleciona e chama Tools
   ↓
5. Tools consultam DataManager e retornam dados
   ↓
6. Agente analisa resultados
   ↓
7. Agente gera resposta em linguagem natural
   ↓
8. Resposta apresentada ao usuário
```

### Decisões de Design

1. **Por que usar Groq?**
   - API rápida e eficiente
   - Excelente para análise de dados estruturados
   - Modelo Mixtral é versátil

2. **Por que LangChain?**
   - Framework consolidado para agentes
   - Facilita integração de tools
   - Suporta múltiplos LLMs

3. **Por que tools em vez de direto em Python?**
   - Agente pode raciocinar sobre qual tool usar
   - Mais flexível para diferentes tipos de perguntas
   - Facilita auditoria do processo decisório

4. **Por que armazenar em memória?**
   - Rápido acesso aos dados
   - Adequado para MVPs e datasets pequeno-médio
   - Pode ser estendido para banco de dados


## 15. Resumo e Próximos Passos

### O que foi implementado:

✅ **Interface A** - Upload de arquivos ZIP com CSVs  
✅ **Interface B** - Consulta em linguagem natural  
✅ **Agente Inteligente** - Baseado em LangChain + Groq  
✅ **5 Tools especializadas** para análise de dados  
✅ **DataManager** para gerenciar múltiplos CSVs  
✅ **Histórico de chat** para contexto contínuo  
✅ **Testes funcionais** com dados de exemplo  

### Como usar:

1. **Configure a chave Groq** na célula 3
2. **Faça upload de um ZIP** na Interface A (células 9-11)
3. **Faça perguntas** na Interface B (células 12-13)
4. **Execute testes** nas células 13-16 para ver exemplos

### Possíveis melhorias:

- [ ] Suporte a banco de dados SQL para datasets maiores
- [ ] Geração automática de gráficos nas respostas
- [ ] Multi-agentes para tarefas complexas
- [ ] Cache de resultados para performance
- [ ] Exportação de respostas em PDF/HTML
- [ ] Suporte a mais tipos de arquivos (Excel, Parquet)
- [ ] Interface web com Streamlit ou FastAPI
